# Evaluate the Logistic Regression model

- Accuracy is generally not a very reliable metric because it can be biased by the most common target class.

There are two other useful metrics:

- _precision_ and
- _recall_.
  
Check the slides for this lesson to get the relevant expressions.

`Precision` is the proportion of positive predictions which are correct. For all flights which are predicted to be delayed, what proportion is actually delayed?

`Recall` is the proportion of positives outcomes which are correctly predicted. For all delayed flights, what proportion is correctly predicted by the model?

The precision and recall are generally formulated in terms of the positive target class. But it's also possible to calculate weighted versions of these metrics which look at both target classes.

The components of the confusion matrix are available as `TN`, `TP`, `FN` and `FP`, as well as the object prediction.

## Instructions

- Find the precision and recall.
- Create a multi-class evaluator and evaluate weighted precision.
- Create a binary evaluator and evaluate AUC using the "areaUnderROC" metric.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flight_manipulate_columns').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [ ]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [2]:
# Read data from CSV file
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M2-Classification/3_LogisticRegression/dataset/flights.csv',
                         sep=',',
                         header=True,
                         inferSchema=True,
                         nullValue='NA')

In [3]:
flights = flights.drop('flight')
flights = flights.dropna()

from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0))\
.drop('mile')\
.withColumn('label', (flights.delay > 15).cast('integer'))

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=[
'mon', 'depart', 'duration'
], outputCol='features')
flights = assembler.transform(flights)

flights = flights.select('mon', 'depart', 'duration', 'features', 'label')

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=43)

# Import the logistic regression class
from pyspark.ml.classification import LogisticRegression

# Create a classifier object and train on training data
logistic = LogisticRegression().fit(flights_train)

# Create predictions for the testing data and show confusion matrix
prediction = logistic.transform(flights_test)

TN = prediction.filter('prediction = 0 AND label = prediction').count()
TP = prediction.filter('prediction = 1 AND label = prediction').count()
FN = prediction.filter('prediction = 0 AND label != prediction').count()
FP = prediction.filter('prediction = 1 AND label != prediction').count()

print("First few predictions from the Logistic Regression model:")

prediction.select('label', 'prediction', 'probability').show(5, False)

First few predictions from the Logistic Regression model:
+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.6207707305791564,0.37922926942084356]|
|1    |0.0       |[0.6207707305791564,0.37922926942084356]|
|1    |0.0       |[0.6708884287948659,0.3291115712051341] |
|1    |0.0       |[0.6007874475395811,0.39921255246041887]|
|0    |0.0       |[0.6219507821102064,0.3780492178897936] |
+-----+----------+----------------------------------------+
only showing top 5 rows



In [4]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Calculate precision and recall
precision = ____
recall = ____
print('precision = {:.2f}\nrecall    = {:.2f}'.format(precision, recall))

# Find weighted precision
multi_evaluator = ____
weighted_precision = multi_evaluator.____(prediction, {multi_evaluator.metricName: "____"})

# Find AUC
binary_evaluator = ____
auc = binary_evaluator.____(____, {____})

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 1862|
|    0|       0.0| 2653|
|    1|       1.0| 2823|
|    0|       1.0| 1947|
+-----+----------+-----+



Now let's unpack that confusion matrix.